db경로와 분석라인 찾기 버튼의 폭을 넓혀서 밑에 외삽길이와 같은 폭으로 만들어주고 프레임을 넓혔을때 db경로 입력창이 넓어지는게 아니고 결과저장폴더 입력창과 세션입력찰의 넓이가 넗어지게 해줘 결과저장 폴더선택도 버튼넓이를 넓혀서 세션삭제와 같은 폭으로 해줘

In [ ]:
from google.colab import drive
import os, subprocess, sys
from pathlib import Path
from google.colab import runtime
from datetime import datetime
import pytz
tz = pytz.timezone('Asia/Seoul')
# 드라이브 마운트
drive.mount("/content/drive", force_remount=True)
start_time = datetime.now(tz)
print(f"🚀 분석 시작 시간: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
# 경로 설정 (필요에 맞게 수정)
ROOT = Path("/content/drive/MyDrive/vm/count_car_ver5.0")   # 프로젝트 루트
VIDEO = Path("/content/drive/MyDrive/video/창원공사중(분석용)/상남사거리 오전첨두.mp4")       # 사용할 영상 경로

# 작업 디렉터리 이동
os.chdir(ROOT)
print("CWD:", Path.cwd())

# 의존성 설치 (최초 1회, 필요 시 재실행)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "colab/requirements_colab.txt"], check=True)
# 실행
cmd = [
    sys.executable, "-u", "-m", "src.pipeline.detect_to_db",
    "--video", str(VIDEO),
    "--config", "colab/app_config_drive.json",
    "--line-settings", "config/lines_sangnam.json",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print("--- stdout ---")
print(proc.stdout)
print("--- stderr ---")
print(proc.stderr)
print("returncode:", proc.returncode)

print("DB created (if run OK):", (ROOT / "output" / "tracks.sqlite"))
end_time = datetime.now(tz)
print(f"🏁 분석 종료 시간: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

# 3. 소요 시간 계산
duration = end_time - start_time
print(f"⏱️ 총 소요 시간: {duration}")

runtime.unassign()


Mounted at /content/drive
🚀 분석 시작 시간: 2026-01-07 14:21:25
CWD: /content/drive/MyDrive/vm/count_car_ver5.0
Running: /usr/bin/python3 -u -m src.pipeline.detect_to_db --video /content/drive/MyDrive/video/창원공사중(분석용)/상남사거리 오전첨두.mp4 --config colab/app_config_drive.json --line-settings config/lines_sangnam.json


In [ ]:
import sqlite3
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

# 1. 데이터베이스 파일 연결 (본인의 실제 마운트 경로에 맞게 수정하세요)

db_path = '/content/drive/MyDrive/vm/count_car_ver5.0/output/tracks.sqlite'
conn = sqlite3.connect(db_path)
cur = conn.cursor()

try:
    # 2. '상남사거리 오전첨두'가 아닌 session_id를 가진 데이터 삭제
    # 'tracks' 테이블과 'track_trajs' 테이블 모두 처리해야 할 수 있습니다.

    print("데이터 삭제 중...")

    # track_trajs 테이블에서 삭제
    cur.execute("DELETE FROM track_trajs WHERE session_id != '상남사거리 오전첨두'")

    # tracks 테이블에서도 해당하지 않는 데이터 삭제 (만약 테이블이 분리되어 있다면)
    # cur.execute("DELETE FROM tracks WHERE session_id != '상남사거리 오전첨두'")

    # 3. 변경 사항 저장
    conn.commit()
    print(f"삭제 완료! 남은 행 수: {cur.rowcount}")

    # 4. 파일 용량 최적화 (실제 삭제된 만큼 파일 크기를 줄임)
    cur.execute("VACUUM")
    print("데이터베이스 최적화(VACUUM) 완료.")

except Exception as e:
    print(f"오류 발생: {e}")
    conn.rollback()

finally:
    conn.close()

Mounted at /content/drive
데이터 삭제 중...
삭제 완료! 남은 행 수: 0
데이터베이스 최적화(VACUUM) 완료.


In [ ]:
from google.colab import drive
import sqlite3, json, zlib
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

DB = Path("/content/drive/MyDrive/vm/count_car_ver5.0/output/tracks.sqlite")
SESSION = "상남사거리 오전첨두"

def decode_traj(blob):
    return json.loads(zlib.decompress(blob).decode("utf-8"))

max_x = max_y = 0.0
with sqlite3.connect(DB) as conn:
    for (blob,) in conn.execute(
        "select traj from track_trajs where session_id=? and traj is not null limit 200",
        (SESSION,),
    ):
        pts = decode_traj(blob)
        for _fid, _tms, x, y in pts:
            if x > max_x: max_x = x
            if y > max_y: max_y = y

print("max_x, max_y =", max_x, max_y)


Mounted at /content/drive
max_x, max_y = 1199.7756958007812 649.5128936767578
